In [3]:
import zarr
import napari
import geff
import networkx as nx
import numpy as np

path = r"C:\Users\larss\Downloads\biohub-cell-tracking-during-development\train\6bba_fe670320.zarr"
path2 = r"C:\Users\larss\Downloads\biohub-cell-tracking-during-development\train\6bba_fe670320.geff"

arr = zarr.open(path, mode='r')["0"]
graph, metadata = geff.read(path2, backend="networkx")

print(graph.number_of_nodes(), "nodes")
print(list(graph.nodes(data=True))[0])  # sanity check property names

# Assign a track_id to each node based on connected components
# (fine for now — doesn't yet split at divisions, see caveat below)
track_id_map = {}
for i, component in enumerate(nx.weakly_connected_components(graph)):
    for node in component:
        track_id_map[node] = i

tracks_data = np.array([
    [track_id_map[n], a["t"], a["z"], a["y"], a["x"]]
    for n, a in graph.nodes(data=True)
])

viewer = napari.Viewer()
viewer.add_image(arr, name="cells")
viewer.add_tracks(tracks_data, name="ground_truth_tracks")
napari.run()

716 nodes
(1000004, {'t': 0, 'z': 55, 'x': 7, 'y': 18})
